# Peak fitting class demo
23/07/25

Demo of `tmo.peakFit` class.

## Load class

Methods:

- Use `xarrays` for data, see the [xarray docs for details](http://xarray.pydata.org/).
- Use `scipy.signal.find_peaks()` for basic peak finding, see the [Scipy docs for details](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html).
- Use `lmfit` for modelling & fitting, see the [lmfit docs for details](https://lmfit.github.io/lmfit-py/intro.html). 
- If present, use `xarray_lmfit` wrapper for fitting, see the [xarray_lmfit docs for details](https://xarray-lmfit.readthedocs.io). (UPDATE 23/07/25: this is now optional, fitting without wrapper also implemented.)

In [1]:
from tmo.peakFit import peakFit

2025-07-23 17:09:01.631 | INFO     | tmo.peakFit:<module>:32 - Using `xarray_lmfit` wrapper.


## Set test data

Currently expects Xarray for dataset.

For quick testing using NIST data: https://itl.nist.gov/div898/strd/nls/data/gauss2.shtml

In [2]:
testDataURL = 'http://itl.nist.gov/div898/strd/nls/data/LINKS/DATA/Gauss2.dat'

# https://itl.nist.gov/div898/strd/nls/data/LINKS/DATA/Gauss2.dat
# import wget
# dataFile = wget.download(testDataURL)
# wget gives 403 error!

# Try requests...
import requests
r = requests.get(testDataURL)

# Import as file (file from wget, string from requests)
import numpy as np
from io import StringIO
with StringIO(r.text) as f:
    dat = np.loadtxt(f, skiprows=60)
    

In [3]:
# Push data to Xarray
import xarray as xr

nistGdata = xr.DataArray(dat[:, 0], coords={'x':dat[:, 1]}, name='nist_gauss2')
# nistGdata

## Quick fit

- Create class object from data.
- Pass any additional model configs.

In [4]:
peaks = peakFit(nistGdata, baseline='ExponentialModel') #, quickFit=False)

2025-07-23 17:09:02.248 | INFO     | tmo.peakFit:findPeaks:98 - Peak finding for thres=66.9126...
2025-07-23 17:09:02.250 | INFO     | tmo.peakFit:findPeaks:108 - Found peaks at [108 153], see `self.peakProps` for more details.


:Overlay
   .Curve.I  :Curve   [x]   (nist_gauss2)
   .VLines.I :VLines   [x]

2025-07-23 17:09:02.444 | INFO     | tmo.peakFit:quickFit:73 - Running quick fit...
2025-07-23 17:09:02.445 | INFO     | tmo.peakFit:createModel:150 - Creating peaks with function GaussianModel...
2025-07-23 17:09:02.450 | INFO     | tmo.peakFit:modelEval:238 - Using params from self.params.
2025-07-23 17:09:02.458 | INFO     | tmo.peakFit:createModel:198 - Created model, self.model=((Model(gaussian, prefix='p0_') + Model(gaussian, prefix='p1_')) + Model(exponential, prefix='base_')).
2025-07-23 17:09:02.459 | INFO     | tmo.peakFit:createModel:201 - Guessed params, set to self.params.
2025-07-23 17:09:02.459 | INFO     | tmo.peakFit:fit:280 - Running lmfit routine with XRlmfit wrapper...
/opt/conda/envs/xrlmfit2025/lib/python3.13/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


:Overlay
   .Curve.Fit   :Curve   [x]   (modelfit_best_fit)
   .Curve.Data  :Curve   [x]   (nist_gauss2)
   .NdOverlay.I :NdOverlay   [components]
      :Curve   [x]   (value)

Here quick fitting is automatically performed, and outputs to the main class.

In [5]:
# Peak find outputs, from scipy.signal.find_peaks
print(peaks.peaks)

peaks.peakProps

[108 153]


{'peak_heights': array([133.8252 ,  96.74503]),
 'prominences': array([84.01965, 20.61717]),
 'left_bases': array([ 67, 132]),
 'right_bases': array([248, 248]),
 'widths': array([36.01019826, 12.79883439]),
 'width_heights': array([91.815375, 86.436445]),
 'left_ips': array([ 87.46528504, 143.89840066]),
 'right_ips': array([123.47548331, 156.69723505])}

In [6]:
# lmfit results
peaks.fitResults

In [7]:
# If xarray_lmfit wrapper is used, there is also a DataSet with results.
peaks.fitDS

<xarray.Dataset> Size: 10kB
Dimensions:                (param: 12, cov_i: 12, cov_j: 12, fit_stat: 8, x: 250)
Coordinates:
  * x                      (x) float64 2kB 1.0 2.0 3.0 4.0 ... 248.0 249.0 250.0
  * param                  (param) <U14 672B 'p0_amplitude' ... 'base_decay'
  * fit_stat               (fit_stat) <U6 192B 'nfev' 'nvarys' ... 'aic' 'bic'
  * cov_i                  (cov_i) <U14 672B 'p0_amplitude' ... 'base_decay'
  * cov_j                  (cov_j) <U14 672B 'p0_amplitude' ... 'base_decay'
Data variables:
    modelfit_results       object 8B <lmfit.model.ModelResult object at 0x750...
    modelfit_coefficients  (param) float64 96B 4.258e+03 107.0 ... 99.02 90.95
    modelfit_stderr        (param) float64 96B 42.38 0.1501 ... 0.5375 1.103
    modelfit_covariance    (cov_i, cov_j) float64 1kB 1.796e+03 2.66 ... 1.217
    modelfit_stats         (fit_stat) float64 64B 218.0 8.0 ... 417.9 446.0
    modelfit_data          (x) float64 2kB 97.59 97.76 96.57 ... 1.183 4.875
    modelfit_best_fit      (x) float64 2kB 97.94 96.86 95.81 ... 6.408 6.338

In [8]:
# lmfit results details via method
# peaks.fitDetails()

## Fitting options

### Model functions

- Basic method `self.createModel()` will automatically add peaks per self.peaks, and a baseline function can also be set.
- Models use lmfit, see https://lmfit.github.io/lmfit-py/builtin_models.html for available models.
- To create a custom model, just configure with lmfit and set to `self.model` to use.

In [9]:
# View model
peaks.model

((Model(gaussian, prefix='p0_') + Model(gaussian, prefix='p1_')) + Model(exponential, prefix='base_'))

In [10]:
# Create model - will automatically add peaks per peaks.peaks
# Models use lmfit, see https://lmfit.github.io/lmfit-py/builtin_models.html
peaks.createModel(model='LorentzianModel', baseline='ExponentialModel')

2025-07-23 17:09:02.982 | INFO     | tmo.peakFit:createModel:150 - Creating peaks with function LorentzianModel...
2025-07-23 17:09:02.987 | INFO     | tmo.peakFit:modelEval:238 - Using params from self.params.
2025-07-23 17:09:02.992 | INFO     | tmo.peakFit:createModel:198 - Created model, self.model=((Model(lorentzian, prefix='p0_') + Model(lorentzian, prefix='p1_')) + Model(exponential, prefix='base_')).
2025-07-23 17:09:02.993 | INFO     | tmo.peakFit:createModel:201 - Guessed params, set to self.params.


In [11]:
peaks.model

((Model(lorentzian, prefix='p0_') + Model(lorentzian, prefix='p1_')) + Model(exponential, prefix='base_'))

### Model parameters

If `self.createModel` uses `guessParams=True` (default case), then `self.params` will be set.

Set to `None` to ignore, or modify independently as required.

Parameters are lmfit class objects, for details see https://lmfit.github.io/lmfit-py/parameters.html

In [12]:
# Currently set parameters
peaks.params

name,value,initial value,min,max,vary,expression
p0_amplitude,40538.8708,40538.870786249994,-inf,inf,True,
p0_center,92.1848739,92.18487394957984,-inf,inf,True,
p0_sigma,81.5000000,81.5,0.00000000,inf,True,
p0_fwhm,163.000000,None,-inf,inf,False,2.0000000*p0_sigma
p0_height,158.330355,None,-inf,inf,False,"0.3183099*p0_amplitude/max(1e-15, p0_sigma)"
p1_amplitude,40538.8708,40538.870786249994,-inf,inf,True,
p1_center,92.1848739,92.18487394957984,-inf,inf,True,
p1_sigma,81.5000000,81.5,0.00000000,inf,True,
p1_fwhm,163.000000,None,-inf,inf,False,2.0000000*p1_sigma
p1_height,158.330355,None,-inf,inf,False,"0.3183099*p1_amplitude/max(1e-15, p1_sigma)"


In [13]:
# Set params using dictionary syntax
# (Note params.update() can also be used, but required params object.)
peaks.params['p0_center'].value = peaks.peaks[0]
peaks.params['p0_amplitude'].value = 10000
peaks.params['p0_amplitude'].min = 0
peaks.params['p1_center'].value = peaks.peaks[1]
peaks.params['p1_center'].vary = False
peaks.params['p1_amplitude'].min = 0

peaks.params

name,value,initial value,min,max,vary,expression
p0_amplitude,10000.0000,40538.870786249994,0.00000000,inf,True,
p0_center,108.000000,92.18487394957984,-inf,inf,True,
p0_sigma,81.5000000,81.5,0.00000000,inf,True,
p0_fwhm,163.000000,None,-inf,inf,False,2.0000000*p0_sigma
p0_height,39.0564294,None,-inf,inf,False,"0.3183099*p0_amplitude/max(1e-15, p0_sigma)"
p1_amplitude,40538.8708,40538.870786249994,0.00000000,inf,True,
p1_center,153.000000,92.18487394957984,-inf,inf,False,
p1_sigma,81.5000000,81.5,0.00000000,inf,True,
p1_fwhm,163.000000,None,-inf,inf,False,2.0000000*p1_sigma
p1_height,158.330355,None,-inf,inf,False,"0.3183099*p1_amplitude/max(1e-15, p1_sigma)"


In [14]:
# Can also use `set` method

# Set method example
peaks.params.set(p0_center={'value':peaks.peaks[0], 'max':500})
peaks.params.set(p1_center={'value':peaks.peaks[1]})

peaks.params

name,value,initial value,min,max,vary,expression
p0_amplitude,10000.0000,40538.870786249994,0.00000000,inf,True,
p0_center,108.000000,108,-inf,500.000000,True,
p0_sigma,81.5000000,81.5,0.00000000,inf,True,
p0_fwhm,163.000000,None,-inf,inf,False,2.0000000*p0_sigma
p0_height,39.0564294,None,-inf,inf,False,"0.3183099*p0_amplitude/max(1e-15, p0_sigma)"
p1_amplitude,40538.8708,40538.870786249994,0.00000000,inf,True,
p1_center,153.000000,153,-inf,inf,False,
p1_sigma,81.5000000,81.5,0.00000000,inf,True,
p1_fwhm,163.000000,None,-inf,inf,False,2.0000000*p1_sigma
p1_height,158.330355,None,-inf,inf,False,"0.3183099*p1_amplitude/max(1e-15, p1_sigma)"


### Manual evaluation

To check adjusted parameters or recalc, use `self.modelEval()`.

Data is pushed to `self.current`.

In [15]:
# Update from current model?
peaks.modelEval()

2025-07-23 17:09:03.056 | INFO     | tmo.peakFit:modelEval:238 - Using params from self.params.


In [16]:
# And plotting...
peaks.plotCurrent()

:Overlay
   .Curve.Current :Curve   [x]   (current)
   .NdOverlay.I   :NdOverlay   [components]
      :Curve   [x]   (value)

In [17]:
# Note data is pushed to self.current
peaks.current

<xarray.DataArray 'current' (x: 250)> Size: 2kB
array([210.17430735, 208.99719657, 207.84589177, 206.72031032,
       205.62037267, 204.5460023 , 203.49712564, 202.47367198,
       201.47557343, 200.5027648 , 199.55518354, 198.63276965,
       197.73546556, 196.86321605, 196.01596811, 195.19367089,
       194.39627549, 193.62373493, 192.87600394, 192.15303885,
       191.45479743, 190.78123877, 190.13232306, 189.50801144,
       188.90826586, 188.33304882, 187.78232321, 187.2560521 ,
       186.75419851, 186.27672519, 185.82359437, 185.39476755,
       184.99020516, 184.60986636, 184.25370876, 183.92168807,
       183.61375784, 183.32986913, 183.06997021, 182.83400617,
       182.62191862, 182.43364532, 182.26911978, 182.12827092,
       182.01102266, 181.91729351, 181.84699617, 181.80003714,
       181.77631623, 181.77572616, 181.79815215, 181.84347141,
       181.91155273, 182.002256  , 182.11543178, 182.2509208 ,
       182.40855353, 182.5881497 , 182.78951786, 183.01245489,
       183.25674558, 183.5221622 , 183.80846402, 184.11539692,
       184.44269295, 184.79006997, 185.15723123, 185.54386502,
       185.94964432, 186.37422644, 186.81725279, 187.27834849,
       187.7571222 , 188.25316583, 188.76605435, 189.2953456 ,
       189.84058014, 190.40128112, 190.97695416, 191.56708729,
...
       198.58113619, 197.1617539 , 195.71628385, 194.24607243,
       192.75247888, 191.2368712 , 189.70062232, 188.14510627,
       186.57169456, 184.98175265, 183.37663664, 181.7576901 ,
       180.12624106, 178.48359929, 176.83105362, 175.16986959,
       173.50128726, 171.82651915, 170.14674853, 168.46312773,
       166.77677678, 165.08878219, 163.40019588, 161.71203434,
       160.02527795, 158.34087042, 156.65971843, 154.98269138,
       153.31062132, 151.64430299, 149.98449391, 148.33191471,
       146.68724946, 145.05114612, 143.42421706, 141.80703973,
       140.20015726, 138.60407927, 137.0192826 , 135.44621218,
       133.88528186, 132.33687536, 130.80134719, 129.27902354,
       127.77020332, 126.27515907, 124.79413797, 123.32736282,
       121.87503298, 120.43732539, 119.01439549, 117.60637819,
       116.2133888 , 114.83552395, 113.47286249, 112.12546638,
       110.79338155, 109.47663873, 108.17525426, 106.88923093,
       105.61855867, 104.36321538, 103.12316757, 101.89837112,
       100.68877189,  99.49430642,  98.31490248,  97.15047975,
        96.00095029,  94.86621918,  93.74618495,  92.64074017,
        91.54977183,  90.47316187,  89.41078757,  88.36252197,
        87.32823427,  86.30779016])
Coordinates:
  * x        (x) float64 2kB 1.0 2.0 3.0 4.0 5.0 ... 247.0 248.0 249.0 250.0
Attributes:
    params:   Parameters([('p0_amplitude', <Parameter 'p0_amplitude', value=1...

### Update fit

Run `self.fit()` to rerun the main fitting routine using any updated parameter and model configs.

In [18]:
peaks.fit()

2025-07-23 17:09:03.287 | INFO     | tmo.peakFit:fit:280 - Running lmfit routine with XRlmfit wrapper...
/opt/conda/envs/xrlmfit2025/lib/python3.13/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


:Overlay
   .Curve.Fit   :Curve   [x]   (modelfit_best_fit)
   .Curve.Data  :Curve   [x]   (nist_gauss2)
   .NdOverlay.I :NdOverlay   [components]
      :Curve   [x]   (value)

## Custom model

- Models use lmfit, see https://lmfit.github.io/lmfit-py/builtin_models.html for available models.
- To create a custom model, just configure with lmfit and set to `self.model` to use.
- Custom models can use lmfit in-built functions, or user-supplied functions, see https://lmfit.github.io/lmfit-py/model.html

In [19]:
# Custom model example
# Code from https://lmfit.github.io/lmfit-py/model.html

from numpy import exp, loadtxt, pi, sqrt
from lmfit import Model

def gaussian(x, amp, cen, wid):
    """1-d gaussian: gaussian(x, amp, cen, wid)"""
    return (amp / (sqrt(2*pi) * wid)) * exp(-(x-cen)**2 / (2*wid**2))


# Create clean class instance, and skip quickFit
peaks = peakFit(nistGdata, baseline='ExponentialModel', quickFit=False)

# Push model to peakFit class
peaks.model = Model(gaussian)

# Reset params
# peaks.params = None
# Or guess
# peaks.params = peaks.model.make_params()
# Or set
peaks.params = peaks.model.make_params(cen=6.5, amp=100, wid=2.0)

# Fit using new model
peaks.fit()

2025-07-23 17:09:03.864 | INFO     | tmo.peakFit:findPeaks:98 - Peak finding for thres=66.9126...
2025-07-23 17:09:03.867 | INFO     | tmo.peakFit:findPeaks:108 - Found peaks at [108 153], see `self.peakProps` for more details.


:Overlay
   .Curve.I  :Curve   [x]   (nist_gauss2)
   .VLines.I :VLines   [x]

2025-07-23 17:09:03.990 | INFO     | tmo.peakFit:fit:280 - Running lmfit routine with XRlmfit wrapper...


:Overlay
   .Curve.Fit  :Curve   [x]   (modelfit_best_fit)
   .Curve.Data :Curve   [x]   (nist_gauss2)
   .Curve.I    :Curve   [x]   (gaussian)

In [20]:
peaks.fitResults

In [21]:
peaks.params

name,value,initial value,min,max,vary
amp,100.000000,100.0,-inf,inf,True
cen,6.50000000,6.5,-inf,inf,True
wid,2.00000000,2.0,-inf,inf,True


In [22]:
# Evaluate and plot current with specific params passed
# Note that setting only some params may give unexpected results here.
peaks.modelEval(cen=50.5, amp=100, wid=2.0)
peaks.plotCurrent()

2025-07-23 17:09:04.254 | INFO     | tmo.peakFit:modelEval:229 - Setting params from passed kwargs.


:Overlay
   .Curve.Current :Curve   [x]   (current)
   .Curve.I       :Curve   [x]   (gaussian)

In [23]:
# Evaluate and plot with self.params

# Update as desired
peaks.params['cen'].value=100

# Eval & plot
peaks.modelEval()
peaks.plotCurrent()

2025-07-23 17:09:04.388 | INFO     | tmo.peakFit:modelEval:238 - Using params from self.params.


:Overlay
   .Curve.Current :Curve   [x]   (current)
   .Curve.I       :Curve   [x]   (gaussian)

## Versions

In [24]:
import scooby
scooby.Report(additional=['tmo', 'xarray', 'jupyter'])

--------------------------------------------------------------------------------
  Date: Wed Jul 23 17:09:04 2025 EDT

                OS : Linux (Ubuntu 20.04)
            CPU(s) : 20
           Machine : x86_64
      Architecture : 64bit
               RAM : 19.5 GiB
       Environment : Jupyter

  Python 3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:27:50) [GCC
  13.3.0]

               tmo : 0.0.1
            xarray : 2025.7.1
           jupyter : Version unknown
             numpy : 2.3.1
             scipy : 1.16.0
           IPython : 9.4.0
        matplotlib : 3.10.3
            scooby : 0.10.1
--------------------------------------------------------------------------------

### Git

In [25]:
# Check current Git commit for local tmo-dev version
from pathlib import Path
!git -C {Path(tmo.__file__).parent} branch
!git -C {Path(tmo.__file__).parent} log --format="%H" -n 1


/bin/bash: -c: line 0: syntax error near unexpected token `('
/bin/bash: -c: line 0: `git -C {Path(tmo.__file__).parent} branch'
/bin/bash: -c: line 0: syntax error near unexpected token `('
/bin/bash: -c: line 0: `git -C {Path(tmo.__file__).parent} log --format="%H" -n 1'


In [26]:
# Check current remote commits
!git ls-remote --heads git://github.com/phockett/tmo-dev


fatal: unable to connect to github.com:
github.com[0: 140.82.113.4]: errno=Connection refused



### Machine & env

In [27]:
%%bash

hostname

conda env list

1fccb2085fdb
# conda environments:
#
base                  *  /opt/conda
xrlmfit2025              /opt/conda/envs/xrlmfit2025

